# Lesson 2.1 - Control Flow and Iteration

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Use `if`, `switch`, and `foreach` to make decisions in scripts
- Iterate over a small dataset and summarize results
- Produce a repeatable status report from script output

**Project file(s) produced:** `scripts/lesson-2.1-control-flow-iteration.ps1` and `artifacts/build-summary.txt`

## Phase 2 Overview

This phase turns ad hoc commands into reusable script logic. You will learn how to branch, loop, accept parameters, handle errors, and return structured output.

### Prerequisites
- Completion of Phase 1
- A working PowerShell installation
- Python 3 with notebook support for the lesson cells

## Concepts

**Control flow** means choosing what happens next. `if` handles one branch, `switch` handles several branches, and loops repeat work for each item in a collection.

In automation, control flow is what turns a static command into a script that can react to success, warning, or failure states.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-2-scripting-mechanics')
artifact_dir = lesson_dir / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Classify build states

This step uses `switch` to turn a list of build states into a simple summary. This is the same shape you would use for deployment results, policy checks, or test outcomes.

In [ ]:
script = textwrap.dedent(r'''
$states = @('passed', 'passed', 'warning', 'failed', 'passed')
$summary = [ordered]@{ Passed = 0; Warning = 0; Failed = 0 }
foreach ($state in $states) {
    switch ($state) {
        'passed'  { $summary.Passed++ }
        'warning' { $summary.Warning++ }
        'failed'  { $summary.Failed++ }
    }
}
$summary.GetEnumerator() | Sort-Object Name | Format-Table -AutoSize
''')
run_powershell(script, 'lesson-2.1-classify-states.ps1')

## Code-Along: Step 2 - Write a summary file

A script becomes more useful when it leaves behind an artifact. Here the artifact is a text file that records the totals so another process could read them later.

In [ ]:
script = textwrap.dedent(r'''
$outputPath = Join-Path $PSScriptRoot 'artifacts/build-summary.txt'
$states = @('passed', 'passed', 'warning', 'failed', 'passed')
$summary = [ordered]@{ Passed = 0; Warning = 0; Failed = 0 }
foreach ($state in $states) {
    switch ($state) {
        'passed'  { $summary.Passed++ }
        'warning' { $summary.Warning++ }
        'failed'  { $summary.Failed++ }
    }
}
$summary.GetEnumerator() | Sort-Object Name | ForEach-Object { "{0}: {1}" -f $_.Name, $_.Value } | Set-Content -Path $outputPath
Get-Content $outputPath
''')
run_powershell(script, 'lesson-2.1-write-summary.ps1')

## Exercise

Extend the state list with a new `skipped` category and update the summary so it counts that state too.

**Your task:**
1. Add the new category to the script.
2. Update the summary collection.
3. Confirm the output file still contains one line per category.

**Expected result:** a text report that reflects the new build state.

In [ ]:
# TODO: add a skipped state and update the summary report
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: extend the summary report"
''')
run_powershell(exercise_script, 'lesson-2.1-exercise.ps1')

## Key Takeaways

- `if`, `switch`, and loops are the decision-making core of scripts.
- A short data list is enough to practice realistic automation logic.
- Writing a summary file makes the script useful to other tools and people.

---

**Next:** [Lesson 2.2 - Functions and Advanced Parameters](./lesson-2.2-functions-advanced-parameters.ipynb)